# XAI Validation

* **Owner:** Pramodya Dewmi
* **Module:** CCS4310 – Deep Learning
* **Project:** Explainable-Fashion-Design-AI

This notebook validates whether the explanations generated by Notebook 1 are technically consistent and reasonably stable, without redoing the full SHAP calculation for the entire dataset. It reads the saved SHAP output and validation/development data, then checks reconstruction consistency, local explanation stability, and the relationship to the actual model inputs.

This notebook intentionally avoids the final test split and keeps its checks on the development/validation data only.

In [1]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

def find_repo_root():
    candidates = [Path.cwd().resolve()]
    candidates.extend(Path.cwd().resolve().parents)
    for p in candidates:
        if (p / 'src').is_dir() and (p / 'data').is_dir() and (p / 'requirements.txt').exists():
            return p
    raise FileNotFoundError('Repository root not found.')

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

INTERIM = ROOT / 'data' / 'interim'
MODELS = ROOT / 'models'
METRICS = ROOT / 'outputs' / 'metrics'
FIGURES = ROOT / 'outputs' / 'figures'
for d in [METRICS, FIGURES]:
    d.mkdir(parents=True, exist_ok=True)

from src.explainability.shap_explainer import (
    check_shap_prediction_consistency,
    explain_row,
    predict_explained_output,
)
from src.explainability.similar_products import find_similar_products, load_embeddings

print('Repository root:', ROOT)
print('Python:', sys.executable)

Repository root: D:\Deep Learning\Project\GITHUB\Explainable-Fashion-Design-AI
Python: d:\Deep Learning\Project\GITHUB\Explainable-Fashion-Design-AI\.venv\Scripts\python.exe


In [2]:
MODEL_CONFIG_PAIRS = [
    ('demand', MODELS / 'demand' / 'best_demand_model.joblib', MODELS / 'demand' / 'demand_model_config.json'),
    ('preference', MODELS / 'preference' / 'best_customer_preference_model.joblib', MODELS / 'preference' / 'customer_preference_model_config.json'),
    ('preference_dev', MODELS / 'preference' / 'best_customer_preference_model_dev.joblib', MODELS / 'preference' / 'customer_preference_model_config_dev.json'),
]
selected_pair = next(
    ((kind, model_path, config_path) for kind, model_path, config_path in MODEL_CONFIG_PAIRS if model_path.is_file() and config_path.is_file()),
    None,
)
if selected_pair is None:
    MODEL_KIND = None
    MODEL_PATH = None
    CONFIG_PATH = None
else:
    MODEL_KIND, MODEL_PATH, CONFIG_PATH = selected_pair

if MODEL_PATH is None or CONFIG_PATH is None:
    print('Saved model/config not found yet. This validation notebook can still be run once the trained pipeline is available.')
    MODEL_AVAILABLE = False
else:
    MODEL_AVAILABLE = True
    import joblib
    pipeline = joblib.load(MODEL_PATH)
    config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
    selected_model = str(config.get('selected_model', '')).strip().lower()
    heuristic_preference_winners = {
        'category preference',
        'global popularity',
        'recent popularity',
    }
    if MODEL_KIND.startswith('preference') and (
        selected_model in heuristic_preference_winners or not hasattr(pipeline, 'named_steps')
    ):
        print('Preference SHAP is unavailable because the selected preference winner is a heuristic baseline.')
        MODEL_AVAILABLE = False
    elif not hasattr(pipeline, 'named_steps'):
        raise TypeError('The saved artifact is not a fitted sklearn Pipeline with prep/model steps; SHAP requires the actual trained pipeline.')
    else:
        run_type = config.get("run_type", "unknown")
        artifact_suffix = "_dev" if run_type != "full" else ""
        required_columns = list(config.get('feature_columns', []))
        categorical_features = list(config.get('categorical_features', []))
        print('Loaded pipeline:', MODEL_PATH.name)
        print('Feature columns:', len(required_columns))

Loaded pipeline: best_demand_model.joblib
Feature columns: 17


In [3]:
if MODEL_AVAILABLE:
    validation_path = INTERIM / 'visuelle_demand_validation.csv'
    if not validation_path.exists():
        print('Validation data missing at', validation_path, '. Skipping validation checks.')
        validation_df = pd.DataFrame()
    else:
        validation_df = pd.read_csv(validation_path, parse_dates=['time'])
        missing = [col for col in required_columns if col not in validation_df.columns]
        if missing:
            raise ValueError(f'Validation split missing required features: {missing}')
        print('Validation rows:', len(validation_df))
        validation_sample = validation_df.sample(n=min(200, len(validation_df)), random_state=42).reset_index(drop=True)
        X_eval = validation_sample[required_columns]

Validation rows: 262417


## 1. SHAP prediction consistency validation

Each sampled row is checked against the actual saved model prediction to ensure that the SHAP reconstruction matches the pipeline prediction, rather than silently accepting a false mismatch.

In [4]:
if MODEL_AVAILABLE and not X_eval.empty:
    consistency_rows = []
    for idx in range(min(10, len(X_eval))):
        consistency = check_shap_prediction_consistency(pipeline, X_eval, row_index=idx, tolerance=1e-3)
        consistency_rows.append(consistency)
    consistency_df = pd.DataFrame(consistency_rows)
    consistency_df['run_type'] = run_type
    display(consistency_df[['row_index', 'actual_prediction', 'reconstructed_prediction', 'difference', 'consistent', 'run_type']])
    print('All sampled SHAP reconstructions consistent within tolerance:', bool((consistency_df['consistent']).all()))
    consistency_df.to_csv(METRICS / f'dewmi_xai_validation{artifact_suffix}.csv', index=False)
else:
    print('Skipping SHAP consistency checks because the saved model or validation sample is unavailable.')

,row_index,actual_prediction,reconstructed_prediction,difference,consistent,run_type
0,0,3.402286,3.402287,7.556000e-07,True,fast_dev
1,1,0.840893,0.840893,1.165435e-07,True,fast_dev
2,2,0.889127,0.889127,1.650266e-07,True,fast_dev
3,3,0.340580,0.340580,1.081639e-07,True,fast_dev
4,4,1.061497,1.061497,2.747175e-07,True,fast_dev
5,5,1.649199,1.649199,2.749435e-07,True,fast_dev
6,6,1.650853,1.650853,1.579372e-07,True,fast_dev
7,7,0.690648,0.690648,8.155277e-08,True,fast_dev
8,8,1.083928,1.083928,1.914084e-07,True,fast_dev
9,9,1.519975,1.519976,5.620453e-07,True,fast_dev


All sampled SHAP reconstructions consistent within tolerance: True


## 2. Local explanation validation and stability checks

We validate local explanations on multiple examples and check whether the top contributors remain reasonably stable under small valid perturbations of a design attribute only when the attribute is explicitly controllable.

In [5]:
if MODEL_AVAILABLE and not X_eval.empty:
    local_examples = []
    for idx in range(min(5, len(X_eval))):
        explanation = explain_row(pipeline, X_eval, row_index=idx)
        top = explanation.top_contributions(n=5, direction='both')
        local_examples.append({
            'row_index': idx,
            'base_value': explanation.base_value,
            'predicted_value': explanation.predicted_value,
            'top_features': '; '.join(top['feature'].head(3).tolist()),
            'top_signs': '; '.join(f'{v:.3f}' for v in top['shap_value'].head(3)),
        })
    display(pd.DataFrame(local_examples))

    row = X_eval.iloc[0].copy()
    mutable_attributes = [attribute for attribute in ['color', 'fabric', 'category'] if attribute in row.index]
    selected_attribute = None
    new_value = None
    for attribute in mutable_attributes:
        observed_values = validation_sample[attribute].dropna().drop_duplicates().tolist()
        different_values = [value for value in observed_values if value != row[attribute]]
        if different_values:
            selected_attribute = attribute
            new_value = different_values[0]
            break

    if selected_attribute is None:
        print('No different valid observed validation value exists for color, fabric, or category; perturbation test is skipped.')
    else:
        perturbed = row.copy()
        old_value = row[selected_attribute]
        perturbed[selected_attribute] = new_value
        original_frame = pd.DataFrame([row], columns=X_eval.columns)
        perturbed_frame = pd.DataFrame([perturbed], columns=X_eval.columns)

        def prediction_score(output):
            if isinstance(output, dict):
                for key in ['prediction', 'predicted_value', 'score']:
                    if key in output:
                        return float(np.asarray(output[key]).reshape(-1)[0])
            for attribute in ['prediction', 'predicted_value', 'score']:
                if hasattr(output, attribute):
                    return float(np.asarray(getattr(output, attribute)).reshape(-1)[0])
            return float(np.asarray(output).reshape(-1)[0])

        original_score = prediction_score(predict_explained_output(pipeline, original_frame))
        perturbed_score = prediction_score(predict_explained_output(pipeline, perturbed_frame))
        original_top = explain_row(pipeline, original_frame, row_index=0).top_contributions(n=5, direction='both')['feature'].tolist()
        perturbed_top = explain_row(pipeline, perturbed_frame, row_index=0).top_contributions(n=5, direction='both')['feature'].tolist()
        original_top_set = set(original_top)
        perturbed_top_set = set(perturbed_top)
        union = original_top_set | perturbed_top_set
        jaccard_overlap = len(original_top_set & perturbed_top_set) / len(union) if union else 1.0

        print('Attribute changed:', selected_attribute)
        print('Old value:', old_value)
        print('New value:', new_value)
        print('Original prediction:', original_score)
        print('Perturbed prediction:', perturbed_score)
        print('Score delta:', perturbed_score - original_score)
        print('Original top SHAP features:', original_top)
        print('Perturbed top SHAP features:', perturbed_top)
        print('Top-feature Jaccard overlap:', jaccard_overlap)
else:
    print('Skipping local explanation validation because the saved model or validation sample is unavailable.')

,row_index,base_value,predicted_value,top_features,top_signs
0,0,1.126854,3.402287,cat__shop_label_12; num__demand_rolling_mean_3...,0.637; 0.523; 0.297
1,1,1.126854,0.840893,num__demand_lag_1; num__demand_rolling_mean_3;...,-0.202; -0.139; 0.081
2,2,1.126854,0.889127,num__demand_rolling_mean_3; num__demand_rollin...,-0.175; -0.056; 0.055
3,3,1.126854,0.340580,num__demand_lag_1; num__demand_lag_3; num__dis...,-0.486; -0.142; -0.140
4,4,1.126854,1.061497,num__demand_rolling_mean_3; num__demand_lag_1;...,-0.175; 0.070; 0.057


Attribute changed: color
Old value: yellow
New value: grey
Original prediction: 3.4022860527038574
Perturbed prediction: 3.410487413406372
Score delta: 0.008201360702514648
Original top SHAP features: ['cat__shop_label_12', 'num__demand_rolling_mean_3', 'cat__category_long dress', 'cat__fabric_tulle', 'num__demand_lag_1']
Perturbed top SHAP features: ['cat__shop_label_12', 'num__demand_rolling_mean_3', 'cat__category_long dress', 'cat__fabric_tulle', 'num__demand_lag_1']
Top-feature Jaccard overlap: 1.0


## 3. Compare global and local explanations

Check that the ranked global importance and the per-row local top features align with the actual model inputs and are not arbitrary because of a mismatched feature mapping.

In [6]:
if MODEL_AVAILABLE and not X_eval.empty:
    global_importance_path = METRICS / f'dewmi_shap_global_importance{artifact_suffix}.csv'
    global_importance = pd.read_csv(global_importance_path, index_col=0) if global_importance_path.exists() else pd.DataFrame()
    if not global_importance.empty:
        top_global = global_importance.head(10).index.tolist()
        ex = explain_row(pipeline, X_eval, row_index=0)
        top_local = ex.top_contributions(n=10, direction='both')['feature'].tolist()
        overlap = set(top_global) & set(top_local)
        print('Top global SHAP features:', top_global)
        print('Top local SHAP features:', top_local)
        print('Overlap between global and local ranked features:', sorted(overlap)[:10])
    else:
        print('Global SHAP CSV missing; compare the notebook outputs after running 05_shap_explainability.ipynb first.')
else:
    print('Skipping global-vs-local comparison because the model or validation sample is unavailable.')

Top global SHAP features: ['num__demand_rolling_mean_3', 'num__demand_lag_1', 'num__restock_past_1', 'num__discount_past_1', 'num__demand_rolling_std_3', 'num__calendar_week', 'cat__season_AW18', 'num__demand_lag_3', 'num__month', 'num__price_past_1']
Top local SHAP features: ['cat__shop_label_12', 'num__demand_rolling_mean_3', 'cat__category_long dress', 'cat__fabric_tulle', 'num__demand_lag_1', 'num__demand_rolling_std_3', 'cat__season_from_time_spring', 'num__discount_past_1', 'cat__season_AW18', 'num__demand_lag_3']
Overlap between global and local ranked features: ['cat__season_AW18', 'num__demand_lag_1', 'num__demand_lag_3', 'num__demand_rolling_mean_3', 'num__demand_rolling_std_3', 'num__discount_past_1']


## 4. Optional similar-product evidence

Similar-product evidence is only used when an actual generated-design query embedding is available. If upstream CLIP/visual embeddings are not ready, the notebook skips gracefully and prints an explanatory message instead of inventing evidence.

In [7]:
embeddings_path = ROOT / 'data' / 'processed' / 'visual_features' / 'deepfashion_clip_embeddings.npy'
metadata_path = ROOT / 'data' / 'processed' / 'visual_features' / 'deepfashion_clip_metadata.csv'
query_embedding = None

if embeddings_path.exists() and metadata_path.exists():
    embeddings, metadata = load_embeddings(embeddings_path, metadata_path)

if query_embedding is None:
    print('Similar-product evidence pending upstream generated-design embedding.')
else:
    similar = find_similar_products(query_embedding, embeddings, metadata, top_k=5, outcome_column='demand' if 'demand' in metadata.columns else None)
    print('Similar-product evidence found:')
    display(similar.head())

Similar-product evidence pending upstream generated-design embedding.
